In [1]:
pip install tensorflow numpy pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import random
from collections import deque
from sklearn.preprocessing import StandardScaler

# Load Dataset
data = pd.read_csv("synthetic_ehr_data.csv")

# Preprocess (Normalize)
scaler = StandardScaler()
data[['Age', 'Glucose', 'HbA1c', 'Systolic_BP', 'Diastolic_BP', 'BMI', 'Exercise']] = scaler.fit_transform(
    data[['Age', 'Glucose', 'HbA1c', 'Systolic_BP', 'Diastolic_BP', 'BMI', 'Exercise']])

# Define States and Actions
states = data.drop(columns=['Action']).values
actions = data['Action'].values
state_size = states.shape[1]
action_size = len(data['Action'].unique())

# DQN Agent Class
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95  # discount rate
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.model = self.build_model()

    def build_model(self):
        model = Sequential([
            Dense(64, activation='relu', input_dim=self.state_size),
            Dropout(0.2),
            Dense(64, activation='relu'),
            Dropout(0.2),
            Dense(self.action_size, activation='linear')
        ])
        model.compile(loss=tf.keras.losses.MeanSquaredError(), optimizer=Adam(learning_rate=self.learning_rate))
        return model


    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.model.predict(state, verbose=0)
        return np.argmax(q_values[0])

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size):
        minibatch = random.sample(self.memory, min(len(self.memory), batch_size))
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target += self.gamma * np.amax(self.model.predict(next_state, verbose=0)[0])
            target_f = self.model.predict(state, verbose=0)
            target_f[0][action] = target
            self.model.fit(state, target_f, epochs=1, verbose=0)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Initialize and train agent
agent = DQNAgent(state_size, action_size)
episodes = 200  # fewer episodes for quicker training
batch_size = 32

for episode in range(episodes):
    index = np.random.randint(len(states))
    state = states[index].reshape(1, -1)
    action = agent.act(state)

    reward = 10 if action == actions[index] else -10
    next_state = state
    done = True

    agent.remember(state, action, reward, next_state, done)
    agent.replay(batch_size)

    if (episode+1) % 50 == 0:
        print(f"Episode {episode+1}/{episodes}, epsilon={agent.epsilon:.2f}")

# Save Model
agent.model.save("dqn_diabetes_model.h5")
print("Model saved successfully.")

c:\Users\chinn\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Episode 50/200, epsilon=0.78
Episode 100/200, epsilon=0.61
Episode 150/200, epsilon=0.47


Episode 200/200, epsilon=0.37
Model saved successfully.
